In [ ]:
# ran on gcolab for free limited gpu
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
%cd "/content/drive/MyDrive/nearest_neighbors/"

/content/drive/MyDrive/nearest_neighbors


In [11]:
import os
import sys
BASE_DIR = os.path.abspath(".")
print(BASE_DIR)
sys.path.append(BASE_DIR)
from collections import Counter
from itertools import chain
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
torch.cuda.empty_cache()
import json
import torch.nn.functional as F
from data_processing.dataloader import Dataloader
from data_processing.dataset import TrainingDataset
from model.query_encoder import QueryEncoder
from model.document_encoder import DocumentEncoder
from model.train import Trainer
%load_ext autoreload
%autoreload 2

/content/drive/MyDrive/nearest_neighbors
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
dataset_path = os.path.join(BASE_DIR, "dataset_analysis/simulation_output/simulation_results_1753448515.json")
dataloader_instance = Dataloader(batch_size=8, dataset_path=dataset_path, test_size=0.2)
train_dataloader = dataloader_instance.get_train_dataloader()
test_dataloader = dataloader_instance.get_test_dataloader()

for batch in train_dataloader:
    print(len(batch))
    print("Keys in the batch:")
    for key in batch.keys():
        print(key)

    print("Shapes of tensors in the batch:")
    for key, tensor in batch.items():
         print(f"{key}: {tensor.shape}")
    break



for batch in test_dataloader:
    print(len(batch))
    print("Keys in the batch:")
    for key in batch.keys():
        print(key)

    print("Shapes of tensors in the batch:")
    for key, tensor in batch.items():
         print(f"{key}: {tensor.shape}")
    break

Dataset split into 204 training and 52 testing samples.
8
Keys in the batch:
query_input_ids
query_attention_mask
positive_document_input_ids
positive_document_attention_mask
positive_numerical_features
negative_document_input_ids
negative_document_attention_mask
negative_numerical_features
Shapes of tensors in the batch:
query_input_ids: torch.Size([8, 128])
query_attention_mask: torch.Size([8, 128])
positive_document_input_ids: torch.Size([8, 128])
positive_document_attention_mask: torch.Size([8, 128])
positive_numerical_features: torch.Size([8, 6])
negative_document_input_ids: torch.Size([8, 128])
negative_document_attention_mask: torch.Size([8, 128])
negative_numerical_features: torch.Size([8, 6])
8
Keys in the batch:
query_input_ids
query_attention_mask
positive_document_input_ids
positive_document_attention_mask
positive_numerical_features
negative_document_input_ids
negative_document_attention_mask
negative_numerical_features
Shapes of tensors in the batch:
query_input_ids: torc

In [13]:
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
query_encoder = QueryEncoder(bert_model_name='bert-base-uncased')
document_encoder = DocumentEncoder(
    bert_model_name='bert-base-uncased',
    numerical_dim=6,
    hidden_dim=128
)

trainer = Trainer(
    query_encoder=query_encoder,
    document_encoder=document_encoder,
    device=device,
    temperature=0.07,
    lr=2e-5,
    save_dir="checkpoints"
)

In [14]:
num_epochs = 5
loss_per_epoch = trainer.train(train_dataloader, num_epochs=num_epochs)
loss_per_epoch = list(enumerate(loss_per_epoch, start=1))
best_epoch, best_loss = min(loss_per_epoch, key=lambda x: x[1])
query_path = f"checkpoints/epoch_{best_epoch}_query_encoder.pt"
doc_path = f"checkpoints/epoch_{best_epoch}_document_encoder.pt"
print(f"\n Best Epoch: {best_epoch} with Loss = {best_loss:.4f}")
print(f"Saved Weights:")
print(f" - Query Encoder: {query_path}")
print(f" - Document Encoder: {doc_path}")

Epoch: 1: 100%|██████████| 26/26 [00:19<00:00,  1.37it/s]


Epoch 1, Average Loss: 0.3009


Epoch: 2: 100%|██████████| 26/26 [00:16<00:00,  1.57it/s]


Epoch 2, Average Loss: 0.2990


Epoch: 3: 100%|██████████| 26/26 [00:16<00:00,  1.62it/s]


Epoch 3, Average Loss: 0.2961


Epoch: 4: 100%|██████████| 26/26 [00:17<00:00,  1.49it/s]


Epoch 4, Average Loss: 0.2963


Epoch: 5: 100%|██████████| 26/26 [00:16<00:00,  1.60it/s]


Epoch 5, Average Loss: 0.2919

 Best Epoch: 5 with Loss = 0.2919
Saved Weights:
 - Query Encoder: checkpoints/epoch_5_query_encoder.pt
 - Document Encoder: checkpoints/epoch_5_document_encoder.pt


In [15]:
trainer.evaluate(test_dataloader)

Evaluating: 100%|██████████| 7/7 [00:01<00:00,  4.27it/s]

Evaluation results:
Average positive similarity: -0.0042
Average negative similarity: 0.0028
Triplet accuracy (pos_sim > neg_sim): 0.4231
Similarity difference statistics:
  Average difference: -0.0070
  Median difference: -0.0000
  Min difference: -0.3581
  Max difference: 0.0005
  Positive differences: 22 (42.31%)
  Zero differences: 0 (0.00%)
  Negative differences: 30 (57.69%)


{'pos_sims': tensor([ 0.0015,  0.0752,  0.0499, -0.0864,  0.0485,  0.0489, -0.0128, -0.0219,
          0.0459,  0.0046,  0.0412, -0.0320,  0.0303, -0.0706, -0.0101, -0.0457,
          0.0237, -0.0374, -0.0100, -0.1696,  0.0532, -0.1732, -0.1153, -0.0513,
          0.0547, -0.0222,  0.0496,  0.0304,  0.0451,  0.0484, -0.0958,  0.0306,
          0.0205,  0.0605, -0.0680,  0.0684,  0.0455,  0.0270,  0.0651, -0.0620,
          0.0156, -0.0469,  0.0293,  0.0621,  0.0439,  0.0268, -0.1655, -0.0858,
         -0.0185,  0.0528,  0.0420, -0.0569]),
 'neg_sims': tensor([ 0.0015,  0.0752,  0.0504, -0.0863,  0.0483,  0.0490, -0.0128, -0.0219,
          0.0460,  0.0045,  0.0414, -0.0320,  0.0306, -0.0706, -0.0096, -0.0454,
          0.0237, -0.0374, -0.0098, -0.1696,  0.0531, -0.1735, -0.1153, -0.0513,
          0.0548, -0.0222,  0.0494,  0.0302,  0.0452,  0.0488, -0.0958,  0.0303,
          0.0205,  0.0605, -0.0679,  0.0682,  0.0455,  0.0269,  0.0679, -0.0626,
          0.0156, -0.0469,  0.0294,  0